In [1]:
!git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git


Cloning into 'unstructuredata_tfidf_repo'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 64 (delta 20), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 30.10 KiB | 7.53 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [2]:
%cd /content/unstructuredata_tfidf_repo
!ls -R

/content/unstructuredata_tfidf_repo
.:
data	 notebooks  requirements.txt  task_to_do.txt
LICENSE  README.md  src		      tests

./data:
documents.py

./notebooks:
tfidf_activity.py

./src:
__init__.py  load_data.py  run_activity.py  tfidf_utils.py

./tests:
test_tfidf_utils.py


In [3]:
!cat src/load_data.py

"""Utilities for loading the classroom dataset."""
from typing import List, Tuple

from data.documents import TITLES, DOCUMENTS


def load_documents() -> Tuple[List[str], List[str]]:
    """Return (titles, documents)."""
    return TITLES, DOCUMENTS


if __name__ == "__main__":
    titles, docs = load_documents()
    print(f"Loaded {len(docs)} documents.")
    for t, d in zip(titles, docs):
        print(f"- {t}: {len(d.split())} words")


In [6]:
!pip install -q scikit-learn pandas numpy pytest



In [7]:
from src.load_data import load_documents

# Load document titles and their corresponding texts
titles, documents = load_documents()

# Check how many documents were loaded
print("Number of titles:", len(titles))
print("Number of documents:", len(documents))

# Display word count for each document
for title, document in zip(titles, documents):
    word_count = len(document.split())
    print(f"{title}: {word_count} words")

# Verify the dataset
assert len(titles) == 10, "Expected 10 titles"
assert len(documents) == 10, "Expected 10 documents"
assert all(len(doc.split()) > 30 for doc in documents), \
    "Every document should contain more than 30 words"

print("\nAll 10 documents loaded successfully!")

Number of titles: 10
Number of documents: 10
AI in Healthcare: 85 words
Climate Change: 83 words
Basketball Strategy: 85 words
Italian Pasta: 91 words
Space Exploration: 86 words
Cybersecurity: 88 words
Music Theory: 87 words
Gardening Vegetables: 89 words
Economics Inflation: 89 words
Public Health: 91 words

All 10 documents loaded successfully!


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words="english")

# Learn the vocabulary and transform documents into numbers
tfidf_matrix = vectorizer.fit_transform(documents)

# Get all unique terms
terms = vectorizer.get_feature_names_out()

# Print matrix information
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(terms))


TF-IDF matrix shape: (10, 537)
Vocabulary size: 537


In [9]:
import pandas as pd

# Convert the sparse matrix into a DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=titles,
    columns=terms
)

print("First 3 documents and first 10 terms:")
display(tfidf_df.iloc[:3, :10])

First 3 documents and first 10 terms:


,ability,accelerate,access,accountability,accuracy,accurate,actions,activity,adaptation,add
AI in Healthcare,0.000000,0.0,0.0,0.1246,0.1246,0.000000,0.000000,0.0,0.000000,0.0
Climate Change,0.000000,0.0,0.0,0.0000,0.0000,0.124795,0.124795,0.0,0.124795,0.0
Basketball Strategy,0.130137,0.0,0.0,0.0000,0.0000,0.000000,0.000000,0.0,0.000000,0.0


In [10]:
for title, row in tfidf_df.iterrows():
    top_terms = row.sort_values(ascending=False).head(5)

    print(f"\n{title}")
    print(top_terms.to_string())


AI in Healthcare
ai              0.373801
healthcare      0.249201
use             0.164779
waiting         0.124600
transforming    0.124600

Climate Change
change       0.249591
emissions    0.249591
energy       0.212175
climate      0.212175
burning      0.124795

Basketball Strategy
decide        0.130137
switch        0.130137
player        0.130137
situations    0.130137
pick          0.130137

Italian Pasta
pasta        0.443095
sauces       0.332321
sauce        0.221547
different    0.188335
tubes        0.110774

Space Exploration
space       0.349043
mars        0.232695
missions    0.232695
agencies    0.116348
advanced    0.116348

Cybersecurity
cybersecurity    0.249431
systems          0.212040
data             0.164931
defenders        0.124716
attackers        0.124716

Music Theory
musicians    0.246952
music        0.246952
practice     0.209932
mix          0.123476
memory       0.123476

Gardening Vegetables
soil        0.358051
year        0.238701
pests       0

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

def search_documents(query):
    # Transform the query using the SAME fitted vectorizer
    query_vector = vectorizer.transform([query])

    # Compare the query with every document
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Rank documents from highest to lowest similarity
    results = pd.DataFrame({
        "Title": titles,
        "Similarity Score": scores
    })

    results = results.sort_values(
        by="Similarity Score",
        ascending=False
    ).reset_index(drop=True)

    return results

In [13]:
queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet"
]

for query in queries:
    print("\nQuery:", query)
    results = search_documents(query)
    display(results)

    print("Top-ranked document:", results.iloc[0]["Title"])


Query: renewable energy and climate policy


,Title,Similarity Score
0,Climate Change,0.323458
1,Economics Inflation,0.095225
2,Gardening Vegetables,0.048458
3,AI in Healthcare,0.000000
4,Basketball Strategy,0.000000
5,Italian Pasta,0.000000
6,Cybersecurity,0.000000
7,Space Exploration,0.000000
8,Music Theory,0.000000
9,Public Health,0.000000


Top-ranked document: Climate Change

Query: password security and phishing


,Title,Similarity Score
0,Cybersecurity,0.176375
1,AI in Healthcare,0.000000
2,Climate Change,0.000000
3,Basketball Strategy,0.000000
4,Italian Pasta,0.000000
5,Space Exploration,0.000000
6,Music Theory,0.000000
7,Gardening Vegetables,0.000000
8,Economics Inflation,0.000000
9,Public Health,0.000000


Top-ranked document: Cybersecurity

Query: healthy sleep exercise and diet


,Title,Similarity Score
0,Public Health,0.284861
1,Gardening Vegetables,0.044702
2,Climate Change,0.000000
3,AI in Healthcare,0.000000
4,Basketball Strategy,0.000000
5,Italian Pasta,0.000000
6,Cybersecurity,0.000000
7,Space Exploration,0.000000
8,Music Theory,0.000000
9,Economics Inflation,0.000000


Top-ranked document: Public Health


In [14]:
# Create a second vectorizer with bigrams and document-frequency limits
bigram_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

try:
    bigram_matrix = bigram_vectorizer.fit_transform(documents)
    bigram_terms = bigram_vectorizer.get_feature_names_out()

    print("Original vocabulary size:", len(terms))
    print("Bigram vocabulary size:", len(bigram_terms))

    bigram_df = pd.DataFrame(
        bigram_matrix.toarray(),
        index=titles,
        columns=bigram_terms
    )

    for title, row in bigram_df.iterrows():
        print(f"\n{title}")
        print(row.sort_values(ascending=False).head(5).to_string())

except ValueError as error:
    print("The optional configuration could not build a vocabulary:")
    print(error)

Original vocabulary size: 537
Bigram vocabulary size: 69

AI in Healthcare
use             0.391487
clean           0.251653
intelligence    0.251653
disease         0.251653
careful         0.251653

Climate Change
climate        0.386225
energy         0.386225
governments    0.193113
efficiency     0.193113
global         0.193113

Basketball Strategy
adjust           0.299666
based            0.299666
defenses         0.299666
decisions        0.299666
communication    0.299666

Italian Pasta
different    0.621057
long         0.310528
improve      0.310528
flavor       0.310528
quality      0.310528

Space Exploration
international    0.501540
medical          0.501540
health           0.438789
data             0.390115
water            0.390115

Cybersecurity
systems      0.434773
data         0.338180
essential    0.217386
defenses     0.217386
personal     0.217386

Music Theory
practice         0.499421
different        0.249710
live             0.249710
communication    0.249

In [15]:
!pytest -q

...                                                                      [100%]
3 passed in 1.35s


In [16]:
newable energy and climate policy": "Climate Change",
    "password security and phishing": "Cybersecurity",
    "healthy sleep exercise and diet": "Public Health"
}

for query, expected_title in expected_results.items():
    results = search_documents(query)
    actual_title = results.iloc[0]["Title"]

    status = "PASS" if actual_title == expected_title else "CHECK"

    print(f"\nQuery: {query}")
    print(f"Expected: {expected_title}")
    print(f"Actual: {actual_title}")
    print(f"Status: {status}")
    print(results.head(3).to_string(index=False))

SyntaxError: unterminated string literal (detected at line 1) (183059657.py, line 1)

In [17]:

expected_results = {
    "renewable energy and climate policy": "Climate Change",
    "password security and phishing": "Cybersecurity",
    "healthy sleep exercise and diet": "Public Health"
}

for query, expected_title in expected_results.items():
    results = search_documents(query)
    actual_title = results.iloc[0]["Title"]

    status = "PASS" if actual_title == expected_title else "CHECK"

    print("\nQuery:", query)
    print("Expected:", expected_title)
    print("Actual:", actual_title)
    print("Status:", status)
    print(results.head(3).to_string(index=False))


Query: renewable energy and climate policy
Expected: Climate Change
Actual: Climate Change
Status: PASS
               Title  Similarity Score
      Climate Change          0.323458
 Economics Inflation          0.095225
Gardening Vegetables          0.048458

Query: password security and phishing
Expected: Cybersecurity
Actual: Cybersecurity
Status: PASS
           Title  Similarity Score
   Cybersecurity          0.176375
AI in Healthcare          0.000000
  Climate Change          0.000000

Query: healthy sleep exercise and diet
Expected: Public Health
Actual: Public Health
Status: PASS
               Title  Similarity Score
       Public Health          0.284861
Gardening Vegetables          0.044702
      Climate Change          0.000000
